# Clasificación de MNIST con PyTorch

Este notebook prepara MNIST, define una red neuronal, la entrena y mide su desempeño. Se utiliza el conjunto de desarrollo para observar el aprendizaje durante el entrenamiento; el conjunto de prueba se consulta al final. El bloque de preparación usa TensorFlow/Keras para descargar MNIST, aunque el modelo y su entrenamiento están implementados en PyTorch.

## Preparación de los datos

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full, test_size=0.20,
    random_state=SEED, stratify=y_train_full
)

x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, "Dev:", x_dev.shape, "Test:", x_test.shape)

TensorFlow: 2.21.0
Train: (48000, 28, 28) Dev: (12000, 28, 28) Test: (10000, 28, 28)


In [6]:
# PyTorch proporciona la red neuronal, los tensores y los lotes de datos.
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Usa la GPU si PyTorch detecta una compatible; de lo contrario utiliza CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Fijar semillas controla la inicialización de pesos y el orden aleatorio de lotes.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

class RedMNISTRegularizada(nn.Module):
    """Red densa para clasificar imágenes MNIST en uno de diez dígitos."""
    def __init__(self):
        super().__init__()
        # Convierte una imagen de forma (1, 28, 28) en un vector de 784 valores.
        self.flatten = nn.Flatten()
        # Capa oculta: 784 características de entrada y 256 unidades aprendibles.
        self.fc1 = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        # Dropout apaga aleatoriamente 30 % de las activaciones solo al entrenar,
        # lo cual ayuda a reducir el sobreajuste.
        self.dropout = nn.Dropout(0.30)
        # Diez salidas, una por clase. Se entregan logits (sin softmax):
        # CrossEntropyLoss incorpora internamente la operación adecuada.
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        """Define el recorrido de una imagen desde la entrada hasta los logits."""
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)

def crear_loader(x, y, batch_size=64, shuffle=False):
    """Empaqueta arreglos NumPy en lotes listos para PyTorch.

    unsqueeze(1) agrega el canal de color requerido: (n, 28, 28) -> (n, 1, 28, 28).
    Las etiquetas se convierten a enteros int64, formato esperado por CrossEntropyLoss.
    shuffle=True mezcla los ejemplos; se activa para entrenamiento y no para evaluación.
    """
    tensores_x = torch.from_numpy(x).float().unsqueeze(1)
    tensores_y = torch.from_numpy(y.astype(np.int64))
    conjunto = TensorDataset(tensores_x, tensores_y)
    return DataLoader(conjunto, batch_size=batch_size, shuffle=shuffle)

# DataLoader entrega mini-lotes; solo el entrenamiento se mezcla en cada época.
train_loader = crear_loader(x_train, y_train, shuffle=True)
dev_loader = crear_loader(x_dev, y_dev)
test_loader = crear_loader(x_test, y_test)

# Se crea la red y se mueve al dispositivo escogido (CPU o GPU).
modelo_torch = RedMNISTRegularizada().to(device)
# La pérdida compara logits con etiquetas enteras 0..9.
criterio = nn.CrossEntropyLoss()
# Adam actualiza los pesos; weight_decay añade una penalización L2 pequeña.
optimizador_torch = torch.optim.Adam(
    modelo_torch.parameters(), lr=0.001, weight_decay=1e-4
)
print("Dispositivo:", device)
print(modelo_torch)

Dispositivo: cpu
RedMNISTRegularizada(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)


In [7]:
def evaluar(loader_datos):
    """Calcula pérdida promedio y accuracy sin actualizar los pesos."""
    # eval() desactiva el comportamiento aleatorio de Dropout.
    modelo_torch.eval()
    suma_perdidas = 0.0
    total_ejemplos = 0
    total_correctos = 0

    # No se necesitan gradientes durante la evaluación: ahorra memoria y tiempo.
    with torch.no_grad():
        for xb, yb in loader_datos:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = modelo_torch(xb)
            perdida = criterio(logits, yb)

            # La pérdida de cada lote es un promedio; se pondera por el tamaño
            # del lote para obtener un promedio correcto incluso en el último lote.
            suma_perdidas += perdida.item() * yb.size(0)
            predicciones = logits.argmax(dim=1)
            total_correctos += (predicciones == yb).sum().item()
            total_ejemplos += yb.size(0)

    return suma_perdidas / total_ejemplos, total_correctos / total_ejemplos

EPOCHS = 5
for epoch in range(EPOCHS):
    # train() activa Dropout y habilita el modo de entrenamiento.
    modelo_torch.train()
    suma_perdidas_train = 0.0
    ejemplos_train = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        # Ciclo estándar: borrar gradientes anteriores, calcular pérdida,
        # propagar gradientes hacia atrás y actualizar parámetros.
        optimizador_torch.zero_grad()
        logits = modelo_torch(xb)
        perdida = criterio(logits, yb)
        perdida.backward()
        optimizador_torch.step()

        suma_perdidas_train += perdida.item() * yb.size(0)
        ejemplos_train += yb.size(0)

    # Se mide el progreso en desarrollo al final de cada época.
    # Estos datos permiten observar generalización sin usar el conjunto de prueba.
    dev_loss, dev_acc = evaluar(dev_loader)
    train_loss = suma_perdidas_train / ejemplos_train
    print(
        f"Época {epoch + 1}/{EPOCHS} - "
        f"train loss: {train_loss:.4f} - "
        f"dev loss: {dev_loss:.4f} - dev accuracy: {dev_acc:.4f}"
    )

Época 1/5 - train loss: 0.3660 - dev loss: 0.1935 - dev accuracy: 0.9472
Época 2/5 - train loss: 0.1747 - dev loss: 0.1347 - dev accuracy: 0.9608
Época 3/5 - train loss: 0.1312 - dev loss: 0.1132 - dev accuracy: 0.9656
Época 4/5 - train loss: 0.1077 - dev loss: 0.0979 - dev accuracy: 0.9716
Época 5/5 - train loss: 0.0891 - dev loss: 0.0956 - dev accuracy: 0.9722


In [8]:
# La prueba final se realiza una vez terminado el entrenamiento.
# No se modifica el modelo con estos resultados; sirven para estimar su desempeño
# en imágenes que no participaron en el entrenamiento ni en la selección del modelo.
test_loss, test_acc = evaluar(test_loader)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

Test loss: 0.0879
Test accuracy: 0.9738
